# Item-Item Similarity Demo
Build a small recommender-style nearest-neighbor lookup with synthetic item vectors.

In [ ]:
%pip install -q vector-engine

## Create the item catalog

In [ ]:
import numpy as np

from vector_engine import VectorArray, VectorIndex

rng = np.random.default_rng(0)
num_items, dim = 300, 64
item_ids = [f"item-{i}" for i in range(num_items)]
item_vectors = rng.normal(size=(num_items, dim)).astype(np.float32)
item_metadata = [
    {"title": f"Item {i}", "category": str(i % 8)}
    for i in range(num_items)
]
items = VectorArray.from_numpy(
    item_vectors, ids=item_ids, metadata=item_metadata, normalize=True
)
print(f"Created {len(item_ids)} items across 8 categories.")

## Find similar items
Exact NumPy search is sufficient for this small catalog and does not require FAISS.

In [ ]:
index = VectorIndex.create(items, metric="cosine", backend="bruteforce")
target_id = "item-42"
query = items.subset([target_id])
results = index.search(query, k=6, return_metadata=True)
neighbors = [
    (item_id, float(score), metadata["title"], metadata["category"])
    for item_id, score, metadata in zip(
        results.ids[0], results.scores[0], results.metadata[0]
    )
    if item_id != target_id
]
for neighbor in neighbors[:5]:
    print(neighbor)